# 08 Pit Stop Analysis

Pit stops are the clearest operational expression of F1 strategy: teams trade track position for tyre performance, and the cost of that trade varies by circuit, weather, and execution quality.

This notebook reads only `data/cleaned/pit.parquet`. Silver EDA must not read Bronze/raw files, because doing so hides a broken Bronze-to-Silver contract.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "08_pit_stop_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def add_event_type(sessions: pd.DataFrame) -> pd.DataFrame:
    sessions = sessions.copy()
    sessions["event_type"] = np.where(
        sessions["session_name"].astype(str).str.lower().eq("sprint"),
        "SPRINT_RACE",
        "GRAND_PRIX_RACE",
    )
    return sessions

def save_fig(fig, name: str):
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

print("=" * 72)
print(f"SILVER STRATEGY EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER STRATEGY EDA - 08_pit_stop_analysis
Start time: 2026-06-02 02:13:43.173360
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


## 1. Pit Data Coverage

Before judging strategy, we verify whether pit stops have been promoted into Silver. A missing Silver pit artifact is not a charting problem; it is a pipeline coverage problem because pit timing is central to tyre strategy features.

In [2]:
pit_path = CLEANED_DATA_PATH / "pit.parquet"
if not pit_path.exists():
    raise FileNotFoundError(
        "Missing Silver artifact: data/cleaned/pit.parquet. "
        "Run the Bronze-to-Silver cleaning step after enabling pit cleaning."
    )
pit = pd.read_parquet(pit_path)

sessions = add_event_type(pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet"))
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
weather = pd.read_parquet(CLEANED_DATA_PATH / "weather.parquet")
starting_grid = pd.read_parquet(CLEANED_DATA_PATH / "starting_grid.parquet")

for column in ["pit_duration", "lane_duration", "stop_duration", "lap_number", "driver_number", "session_key"]:
    if column in pit.columns:
        pit[column] = pd.to_numeric(pit[column], errors="coerce")
if "date" in pit.columns:
    pit["date"] = pd.to_datetime(pit["date"], errors="coerce", utc=True)

coverage = pd.DataFrame([{
    "pit_source": "silver",
    "silver_pit_exists": pit_path.exists(),
    "rows": len(pit),
    "sessions": pit["session_key"].nunique() if "session_key" in pit.columns and len(pit) else 0,
    "drivers": pit["driver_number"].nunique() if "driver_number" in pit.columns and len(pit) else 0,
    "coverage_issue": None,
}])
coverage.to_csv(OUTPUT_TABLES / "pit_coverage.csv", index=False)
display(coverage)

,pit_source,silver_pit_exists,rows,sessions,drivers,coverage_issue
0,silver,True,1891,64,31,None


In [3]:
fig = px.bar(
    coverage,
    x="pit_source",
    y="rows",
    color="silver_pit_exists",
    title="Pit Stop Coverage Source",
    text="rows",
)
save_fig(fig, "pit_coverage")

## 2. Pit Duration Distribution

Pit duration combines lane loss and stationary servicing behavior depending on source semantics. We therefore profile `pit_duration`, `lane_duration`, and `stop_duration` separately when present.

In [4]:
duration_cols = [column for column in ["pit_duration", "lane_duration", "stop_duration"] if column in pit.columns]
duration_summary = pit[duration_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3).reset_index().rename(columns={"index": "metric"}) if duration_cols else pd.DataFrame()
duration_summary.to_csv(OUTPUT_TABLES / "pit_duration_summary.csv", index=False)
display(duration_summary)

pit_long = pit[duration_cols].melt(var_name="duration_type", value_name="seconds").dropna() if duration_cols else pd.DataFrame()
pit_long = pit_long[pit_long["seconds"].between(0, 180)]
fig = px.histogram(
    pit_long,
    x="seconds",
    color="duration_type",
    nbins=60,
    barmode="overlay",
    title="Pit Stop Duration Distribution",
)
save_fig(fig, "pit_duration_distribution")

,metric,pit_duration,lane_duration,stop_duration
0,count,1821.000,1821.000,967.000
1,mean,24.370,24.370,3.807
2,std,6.572,6.572,3.483
3,min,12.800,12.800,1.800
4,5%,17.900,17.900,2.100
5,25%,21.680,21.680,2.400
6,50%,23.284,23.284,2.700
7,75%,25.500,25.500,3.400
8,95%,32.800,32.800,11.100
9,max,155.100,155.100,38.300


## 3. Pit Windows and Team Execution

The lap of the first stop often reveals the tyre strategy window. Team ranking is joined on `session_key + driver_number` to avoid mixing drivers across sessions when seat/team changes occur.

In [5]:
pit_enriched = pit.merge(
    drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"]),
    on=["session_key", "driver_number"],
    how="left",
)
pit_enriched = pit_enriched.merge(
    sessions[["session_key", "year", "event_type", "circuit_short_name", "country_name"]],
    on="session_key",
    how="left",
)
pit_enriched["valid_pit_duration"] = pit_enriched["pit_duration"].where(pit_enriched["pit_duration"].between(5, 120)) if "pit_duration" in pit_enriched.columns else np.nan

pit_window = pit_enriched.dropna(subset=["lap_number"]).groupby(["event_type", "lap_number"], as_index=False).size().rename(columns={"size": "pit_stops"})
pit_window.to_csv(OUTPUT_TABLES / "pit_window_by_lap.csv", index=False)
display(pit_window.sort_values("pit_stops", ascending=False).head(15))

fig = px.line(
    pit_window,
    x="lap_number",
    y="pit_stops",
    color="event_type",
    title="Pit Stop Timing Window by Lap",
)
save_fig(fig, "pit_window_by_lap")

,event_type,lap_number,pit_stops
31,GRAND_PRIX_RACE,32,73
11,GRAND_PRIX_RACE,12,57
10,GRAND_PRIX_RACE,11,56
0,GRAND_PRIX_RACE,1,56
27,GRAND_PRIX_RACE,28,55
28,GRAND_PRIX_RACE,29,51
26,GRAND_PRIX_RACE,27,49
36,GRAND_PRIX_RACE,37,47
32,GRAND_PRIX_RACE,33,45
6,GRAND_PRIX_RACE,7,44


In [6]:
team_pit = pit_enriched.dropna(subset=["valid_pit_duration", "team_name"]).groupby("team_name", as_index=False).agg(
    avg_pit_duration=("valid_pit_duration", "mean"),
    median_pit_duration=("valid_pit_duration", "median"),
    pit_count=("valid_pit_duration", "count"),
    p95_pit_duration=("valid_pit_duration", lambda s: s.quantile(0.95)),
)
team_pit = team_pit[team_pit["pit_count"] >= 10].sort_values("median_pit_duration")
team_pit.to_csv(OUTPUT_TABLES / "team_pit_performance.csv", index=False)
display(team_pit)

fig = px.bar(
    team_pit,
    x="median_pit_duration",
    y="team_name",
    orientation="h",
    color="pit_count",
    title="Team Pit Execution Ranking",
)
save_fig(fig, "team_pit_performance")

,team_name,avg_pit_duration,median_pit_duration,pit_count,p95_pit_duration
9,RB,25.013195,22.7000,77,32.78460
4,Ferrari,23.409215,22.8910,181,30.50000
10,Racing Bulls,23.212282,22.8990,85,30.11760
11,Red Bull Racing,23.921873,22.9740,189,32.50580
7,McLaren,23.530311,23.0240,177,30.61320
8,Mercedes,23.356516,23.0320,184,30.04335
0,Alpine,24.234041,23.3410,170,31.21260
6,Kick Sauber,25.607700,23.4475,170,35.90000
1,Aston Martin,24.749121,23.5320,190,34.22780
5,Haas F1 Team,25.256827,23.5620,185,37.33040


## 4. Strategic Impact

Pit performance alone rarely explains the race result. We combine pit count, average pit duration, grid position, finish position, and weather to separate operational execution from strategic context.

In [7]:
driver_pit = pit_enriched.groupby(["session_key", "driver_number"], as_index=False).agg(
    pit_count=("lap_number", "count"),
    avg_pit_duration=("valid_pit_duration", "mean"),
    first_pit_lap=("lap_number", "min"),
    last_pit_lap=("lap_number", "max"),
)
finish = session_result[["session_key", "driver_number", "position", "points", "dnf", "dns", "dsq"]].copy()
finish["finish_pos"] = pd.to_numeric(finish["position"], errors="coerce")
grid = starting_grid[["session_key", "driver_number", "position"]].rename(columns={"position": "grid_pos"})
grid["grid_pos"] = pd.to_numeric(grid["grid_pos"], errors="coerce")
session_weather = weather.groupby("session_key", as_index=False).agg(
    rainfall=("rainfall", "max"),
    track_temperature=("track_temperature", "mean"),
)
pit_impact = (
    driver_pit
    .merge(finish, on=["session_key", "driver_number"], how="left")
    .merge(grid, on=["session_key", "driver_number"], how="left")
    .merge(session_weather, on="session_key", how="left")
)
pit_impact["positions_gained"] = pit_impact["grid_pos"] - pit_impact["finish_pos"]
pit_impact.to_csv(OUTPUT_TABLES / "pit_strategy_impact.csv", index=False)
display(pit_impact.head(20))

impact_corr = pit_impact[["pit_count", "avg_pit_duration", "first_pit_lap", "positions_gained", "finish_pos", "track_temperature", "rainfall"]].corr()
impact_corr.to_csv(OUTPUT_TABLES / "pit_impact_correlations.csv")
fig = px.imshow(
    impact_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Pit Strategy Impact Correlation Matrix",
)
save_fig(fig, "pit_strategy_impact_correlation")

,session_key,driver_number,pit_count,avg_pit_duration,first_pit_lap,last_pit_lap,position,points,dnf,dns,dsq,finish_pos,grid_pos,rainfall,track_temperature,positions_gained
0,9472,1,2,24.600000,17,37,1.0,26.0,False,False,False,1.0,1,0.0,23.652866,0.0
1,9472,2,3,28.666667,10,40,20.0,0.0,False,False,False,20.0,18,0.0,23.652866,-2.0
2,9472,3,2,24.850000,13,35,13.0,0.0,False,False,False,13.0,14,0.0,23.652866,1.0
3,9472,4,2,24.450000,13,33,6.0,8.0,False,False,False,6.0,7,0.0,23.652866,1.0
4,9472,10,3,27.166667,12,43,18.0,0.0,False,False,False,18.0,20,0.0,23.652866,2.0
5,9472,11,2,24.500000,12,36,2.0,18.0,False,False,False,2.0,5,0.0,23.652866,3.0
6,9472,14,2,24.900000,15,41,9.0,2.0,False,False,False,9.0,6,0.0,23.652866,-3.0
7,9472,16,2,24.050000,11,34,4.0,12.0,False,False,False,4.0,2,0.0,23.652866,-2.0
8,9472,18,2,24.450000,9,27,10.0,1.0,False,False,False,10.0,12,0.0,23.652866,2.0
9,9472,20,2,25.150000,11,32,12.0,0.0,False,False,False,12.0,15,0.0,23.652866,3.0


In [8]:
weather_pit = pit_enriched.merge(session_weather, on="session_key", how="left")
weather_pit["condition"] = np.where(weather_pit["rainfall"].fillna(0) > 0, "Wet session", "Dry session")
weather_summary = weather_pit.dropna(subset=["valid_pit_duration"]).groupby("condition", as_index=False).agg(
    pit_stops=("valid_pit_duration", "count"),
    avg_pit_duration=("valid_pit_duration", "mean"),
    median_pit_duration=("valid_pit_duration", "median"),
)
weather_summary.to_csv(OUTPUT_TABLES / "pit_weather_summary.csv", index=False)
display(weather_summary)

fig = px.box(
    weather_pit.dropna(subset=["valid_pit_duration"]),
    x="condition",
    y="valid_pit_duration",
    color="condition",
    title="Pit Duration Under Dry vs Wet Sessions",
)
save_fig(fig, "pit_weather_impact")

,condition,pit_stops,avg_pit_duration,median_pit_duration
0,Dry session,1318,24.575042,23.2905
1,Wet session,502,23.572446,23.2715


## Final Pit Stop Report

In [9]:
fastest_team = team_pit.iloc[0].to_dict() if len(team_pit) else {}
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": "PASS",
    "pit_source": "silver",
    "pit_rows": int(len(pit)),
    "sessions_with_pit": int(pit["session_key"].nunique()) if "session_key" in pit.columns and len(pit) else 0,
    "median_pit_duration": float(pit_enriched["valid_pit_duration"].median()) if "valid_pit_duration" in pit_enriched.columns else 0.0,
    "fastest_team": fastest_team.get("team_name"),
    "fastest_team_median_duration": float(fastest_team.get("median_pit_duration", 0.0)) if fastest_team else 0.0,
    "coverage_issue": None,
}
write_report("pit_stop_analysis", report)
write_insight(
    "Silver Pit Stop Analysis Insights",
    [
        f"Analyzed {report['pit_rows']:,} pit records from {report['pit_source']}.",
        f"Median valid pit duration: {report['median_pit_duration']:.2f}s.",
        f"Fastest team by median duration: {report['fastest_team']}.",
    ],
    [],
    [
        "Keep pit as a required Silver artifact; never fallback to raw files inside Silver EDA.",
        "Use first_pit_lap, pit_count, and team rolling pit duration as Gold strategy features.",
        "Treat wet sessions separately because pit timing and tyre calls change under rain risk.",
    ],
)
(CHECKPOINTS / "silver_pit_stop_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '08_pit_stop_analysis', 'timestamp': '2026-06-02T02:13:45.759778', 'status': 'PASS', 'pit_source': 'silver', 'pit_rows': 1891, 'sessions_with_pit': 64, 'median_pit_duration': 23.282, 'fastest_team': 'RB', 'fastest_team_median_duration': 22.7, 'coverage_issue': None}
